## Search Engine with Tools and Agents

In [32]:
# Arxiv---> For Scientific Research Papers
## Tools Creation

from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper




In [33]:
# Using wikipedia tool (resilient wrapper)
# Wikipedia often returns empty/HTML bodies without a User-Agent / when rate-limited,
# which makes the default WikipediaAPIWrapper raise JSONDecodeError.
import time
import wikipedia
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

wikipedia.set_user_agent("QandA-ChatBot/1.0 (educational; LangChain learner)")


class WikiInput(BaseModel):
    query: str = Field(description="query to look up on wikipedia")


def safe_wikipedia(query: str) -> str:
    last_error = None
    for attempt in range(3):
        try:
            titles = wikipedia.search(query, results=1)
            if not titles:
                return "No good Wikipedia Search Result was found"
            summary = wikipedia.summary(titles[0], sentences=4, auto_suggest=False)
            return f"Page: {titles[0]}\nSummary: {summary[:300]}"
        except Exception as e:
            last_error = e
            time.sleep(1.5 * (attempt + 1))
    return f"Wikipedia lookup failed after retries: {type(last_error).__name__}: {last_error}"


wiki_tool = StructuredTool.from_function(
    func=safe_wikipedia,
    name="wikipedia",
    description=(
        "A wrapper around Wikipedia. Useful for general questions about people, "
        "places, companies, historical events, or other subjects. Input should be a search query."
    ),
    args_schema=WikiInput,
)
wiki_tool.name


'wikipedia'

In [34]:
# Arxiv tool compatible with arxiv>=4.x
# (langchain_community ArxivAPIWrapper still calls Search.results(), which was removed)
import arxiv
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field


class ArxivInput(BaseModel):
    query: str = Field(
        description="ArXiv search query or paper id, e.g. '1706.03762' or 'attention is all you need'"
    )


def safe_arxiv(query: str) -> str:
    try:
        client = arxiv.Client()
        # Prefer id lookup when query looks like an arXiv id
        cleaned = query.strip()
        if cleaned.replace(".", "").replace("v", "").replace("V", "").replace(" ", "").isdigit() or (
            len(cleaned.split(".")) == 2 and cleaned.replace(".", "").isdigit()
        ) or cleaned.replace(".", "").replace("v", "")[:4].isdigit():
            search = arxiv.Search(id_list=[cleaned.split("v")[0].split("V")[0]], max_results=1)
        else:
            search = arxiv.Search(query=cleaned, max_results=1, sort_by=arxiv.SortCriterion.Relevance)

        results = list(client.results(search))
        if not results:
            return "No good ArXiv Search Result was found"

        paper = results[0]
        summary = (paper.summary or "").replace("\n", " ")[:300]
        authors = ", ".join(a.name for a in paper.authors[:5])
        return (
            f"Title: {paper.title}\n"
            f"Authors: {authors}\n"
            f"Published: {paper.published.date() if paper.published else 'N/A'}\n"
            f"Entry ID: {paper.entry_id}\n"
            f"Summary: {summary}"
        )
    except Exception as e:
        return f"ArXiv lookup failed: {type(e).__name__}: {e}"


arxiv_tool = StructuredTool.from_function(
    func=safe_arxiv,
    name="arxiv",
    description=(
        "A wrapper around Arxiv.org. Useful for answering questions about physics, "
        "math, computer science, and other scientific papers. Input should be a search "
        "query or an arXiv paper id like 1706.03762."
    ),
    args_schema=ArxivInput,
)
arxiv_tool.name



'arxiv'

In [35]:
## Custom Tools
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [36]:
loader=WebBaseLoader("https://docs.langchain.com/")
docs=loader.load()
documents= RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectorstore=FAISS.from_documents(documents,OpenAIEmbeddings())
retriever=vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002098170EAA0>, search_kwargs={})

In [37]:
from langchain_core.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "langchain-search",
    "Search the LangChain documentation",
)
retriever_tool.name

'langchain-search'

In [38]:
tools=[wiki_tool,arxiv_tool, retriever_tool]

In [39]:
# Run all these tools with LLM Models and Agents
# Tools, LLM--> AgentExecutor
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-120b", api_key=groq_api_key)
llm


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002098170FE50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002098170E6B0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [40]:
# Prompt template (LangChain 1.x: use LangSmith SDK instead of langchain.hub)
from langsmith import Client

prompt = Client().pull_prompt(
    "hwchase17/openai-functions-agent",
    include_model=False,
    dangerously_pull_public_prompt=True,
)
prompt.messages


[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [41]:
# Agents
from langchain_classic.agents import create_openai_tools_agent

agent = create_openai_tools_agent(llm, tools, prompt)
agent


RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

In [42]:
# Agent Executor
from langchain_classic.agents import AgentExecutor

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)

agent_executor.invoke({"input": "What is the capital of France?"})




> Entering new AgentExecutor chain...
The capital of France is **Paris**.

> Finished chain.


{'input': 'What is the capital of France?',
 'output': 'The capital of France is **Paris**.'}

In [43]:
agent_executor.invoke({"input": "What is Machine Learning?"})



> Entering new AgentExecutor chain...
**Machine Learning (ML)** is a subfield of artificial intelligence that focuses on developing algorithms and statistical models that enable computers to **learn from data** and make predictions or decisions without being explicitly programmed for each specific task.

### Core Idea
- **Learning from Experience:** Just as humans improve their performance on a task by gaining experience, an ML system improves its performance by analyzing examples (training data).
- **Generalization:** The goal is to capture patterns in the training data that can be applied to new, unseen data.

### How It Works
1. **Data Collection:** Gather a dataset that represents the problem domain (e.g., images, text, sensor readings).
2. **Model Selection:** Choose a mathematical model (e.g., linear regression, decision tree, neural network) that can map inputs to outputs.
3. **Training:** Use an optimization algorithm (e.g., gradient descent) to adjust the model’s parameters 

{'input': 'What is Machine Learning?',
 'output': '**Machine Learning (ML)** is a subfield of artificial intelligence that focuses on developing algorithms and statistical models that enable computers to **learn from data** and make predictions or decisions without being explicitly programmed for each specific task.\n\n### Core Idea\n- **Learning from Experience:** Just as humans improve their performance on a task by gaining experience, an ML system improves its performance by analyzing examples (training data).\n- **Generalization:** The goal is to capture patterns in the training data that can be applied to new, unseen data.\n\n### How It Works\n1. **Data Collection:** Gather a dataset that represents the problem domain (e.g., images, text, sensor readings).\n2. **Model Selection:** Choose a mathematical model (e.g., linear regression, decision tree, neural network) that can map inputs to outputs.\n3. **Training:** Use an optimization algorithm (e.g., gradient descent) to adjust the

In [44]:
agent_executor.invoke({"input": "Tell me about Virat Kohli?"})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Virat Kohli'}`


Page: Virat Kohli
Summary: Virat Kohli (born 5 November 1988) is an Indian international cricketer and former all-format captain of the Indian national cricket team. He is a right-handed batter and an occasional right-arm medium-pace bowler. Considered one of the greatest batters in limited-overs cricket, he has been acclaime**Virat Kohli – A Cricketing Phenomenon**

| **Basic Details** | |
|-------------------|---|
| **Full Name** | Virat Kohli |
| **Date of Birth** | 5 November 1988 (age 38) |
| **Place of Birth** | Delhi, India |
| **Height** | 1.80 m (5 ft 11 in) |
| **Batting Style** | Right‑hand bat |
| **Bowling Style** | Right‑arm medium (occasional) |
| **Primary Role** | Top‑order batter |
| **International Debut** | ODI – 18 August 2008 vs Sri Lanka (Johannesburg) <br> T20I – 12 June 2010 vs Zimbabwe (Harare) <br> Test – 20 June 2011 vs West Indies (Barbados) |
| **Captaincy** | 

{'input': 'Tell me about Virat Kohli?',
 'output': '**Virat Kohli – A Cricketing Phenomenon**\n\n| **Basic Details** | |\n|-------------------|---|\n| **Full Name** | Virat Kohli |\n| **Date of Birth** | 5\u202fNovember\u202f1988 (age\u202f38) |\n| **Place of Birth** | Delhi, India |\n| **Height** | 1.80\u202fm (5\u202fft\u202f11\u202fin) |\n| **Batting Style** | Right‑hand bat |\n| **Bowling Style** | Right‑arm medium (occasional) |\n| **Primary Role** | Top‑order batter |\n| **International Debut** | ODI – 18\u202fAugust\u202f2008 vs\u202fSri Lanka (Johannesburg) <br> T20I – 12\u202fJune\u202f2010 vs\u202fZimbabwe (Harare) <br> Test – 20\u202fJune\u202f2011 vs\u202fWest\u202fIndies (Barbados) |\n| **Captaincy** | India (Test: 2014‑2022, ODI & T20I: 2017‑2022) |\n| **Current Teams** | Rajasthan Royals (IPL), Delhi (domestic), India (un‑capped) |\n\n---\n\n## 1. Early Life & Background\n- **Family** – Son of Prem and Saroj Kohli. His father was a former club‑level cricketer who introdu

In [45]:
agent_executor.invoke({"input": "What is 1706.03762 this research paper is all aboutt?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones
Published: 2017-06-12
Entry ID: http://arxiv.org/abs/1706.03762v7
Summary: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer**Paper:** *Attention Is All You Need* (arXiv:1706.03762)  
**Authors:** Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin  
**Published:** June 2017 (original version), later updated to v7 (Oct 2019)  

---

## 1. What the paper proposes

The authors introduce a **new neural network architecture for sequence‑to‑sequence tasks called the *Transfo

{'input': 'What is 1706.03762 this research paper is all aboutt?',
 'output': '**Paper:** *Attention Is All You Need* (arXiv:1706.03762)  \n**Authors:** Ashish\u202fVaswani, Noam\u202fShazeer, Niki\u202fParmar, Jakob\u202fUszkoreit, Llion\u202fJones, Aidan\u202fN.\u202fGomez, Łukasz\u202fKaiser, Illia\u202fPolosukhin  \n**Published:** June\u202f2017 (original version), later updated to v7 (Oct\u202f2019)  \n\n---\n\n## 1. What the paper proposes\n\nThe authors introduce a **new neural network architecture for sequence‑to‑sequence tasks called the *Transformer***.  \nUnlike the dominant models of the time—RNNs (LSTMs/GRUs) and CNNs—the Transformer **contains no recurrence or convolution**. Its entire computation is built from **attention mechanisms** and simple feed‑forward layers.\n\n### Core ideas\n| Idea | Description |\n|------|-------------|\n| **Self‑attention (scaled dot‑product attention)** | For each position in a sequence, the model computes a weighted sum of *all* other posit